In [1]:
import pandas as pd
import numpy as np

In [3]:
bls = pd.read_csv(r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\Supplementary Data\bls_healthcare_grouped.csv")
bls.head()

,year,state_code,occ_code,occ_title,tot_emp,a_mean,occupation_group
0,2011,AL,29-1011,Chiropractors,230.0,75320.0,Practitioner
1,2011,AL,29-1021,"Dentists, General",820.0,205130.0,Other
2,2011,AL,29-1023,Orthodontists,70.0,204560.0,Other
3,2011,AL,29-1029,"Dentists, All Other Specialists",50.0,193120.0,Other
4,2011,AL,29-1031,Dietitians and Nutritionists,650.0,49540.0,Practitioner


In [ ]:
"""
Step A: Why we aggregate first

Hospitals don’t employ one occupation — they operate in a state labor market.

So we compute group-level wages like:

Physician wage in Alabama, 2011
RN wage in Alabama, 2011

That’s why we use:

∑
(
a_mean
×
tot_emp
)
∑
(
tot_emp
)
∑(tot_emp)
∑(a_mean×tot_emp)
"""

In [4]:
# Weighted wage
bls["weighted_wage"] = bls["a_mean"] * bls["tot_emp"]

# Aggregate to state–year–group
bls_grouped = (
    bls
    .groupby(["state_code", "year", "occupation_group"])
    .agg(
        total_wage=("weighted_wage", "sum"),
        total_emp=("tot_emp", "sum")
    )
    .reset_index()
)

# Final average wage per group
bls_grouped["avg_wage"] = (
    bls_grouped["total_wage"] / bls_grouped["total_emp"]
)

# Keep only what we need
bls_long = bls_grouped[
    ["state_code", "year", "occupation_group", "avg_wage"]
]

In [5]:
bls_wide = (
    bls_long
    .pivot_table(
        index=["state_code", "year"],
        columns="occupation_group",
        values="avg_wage"
    )
    .reset_index()
)

bls_wide.columns.name = None
bls_wide = bls_wide.rename(columns={
    "Physician": "wage_physician",
    "RN": "wage_rn",
    "Practitioner": "wage_practitioner",
    "Support": "wage_support"
})

In [6]:
bls_wide.head()

,state_code,year,Other,wage_physician,wage_practitioner,wage_rn,wage_support
0,AK,2011,82926.247423,138086.285714,93555.449438,82080.0,37513.439099
1,AK,2012,82582.447818,136453.564356,95827.260274,80970.0,38182.325581
2,AK,2013,86809.962617,121999.130435,100306.978723,83640.0,37542.847425
3,AK,2014,92538.439201,120180.000000,95246.052632,85740.0,38367.525880
4,AK,2015,94885.119266,130628.000000,101273.491525,88510.0,40070.605128


In [7]:
hospital = pd.read_csv(r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\ML Modelling\hospital_kpi_cms_census_rucc.csv")
hospital.head()

,year,state_code,provider_type,ccn_facility_type,number_of_beds,total_bed_days_available,occupancy_rate,total_discharges__v___xviii___xix___unknown_,total_days__v___xviii___xix___unknown_,fte___employees_on_payroll,...,uncompensated_care_percent,charity_care_percent,rucc_code,rural_urban,fips,population,median_age,median_income,poverty_rate,higher_education_rate
0,2011,AL,General Short-Term (includes CAH),Short-Term Hospital,114.0,41610.0,0.472026,5283.0,19641.0,598.72,...,2.482716,2.263904,4.0,Rural,1095.0,NaN,NaN,NaN,NaN,NaN
1,2011,MT,General Short-Term (includes CAH),Critical Access Hospital,25.0,9125.0,0.131068,155.0,1196.0,70.56,...,9.372838,2.320308,3.0,Urban,30009.0,NaN,NaN,NaN,NaN,NaN
2,2011,AL,General Short-Term (includes CAH),Short-Term Hospital,46.0,16790.0,0.159678,892.0,2681.0,72.65,...,15.092862,10.117954,6.0,Rural,1123.0,NaN,NaN,NaN,NaN,NaN
3,2011,AL,Rehabilitation Hospital,Rehabilitation Hospital,100.0,36500.0,0.887068,2390.0,32378.0,297.77,...,-0.031671,4.613961,1.0,Urban,1073.0,NaN,NaN,NaN,NaN,NaN
4,2011,FL,Rehabilitation Hospital,Rehabilitation Hospital,70.0,25550.0,0.663836,1370.0,16961.0,154.60,...,-0.090532,7.756714,1.0,Urban,12103.0,NaN,NaN,NaN,NaN,NaN


In [8]:
hospital = hospital.merge(
    bls_wide,
    on=["state_code", "year"],
    how="left",
    validate="m:1"
)

In [10]:
hospital.head()

,year,state_code,provider_type,ccn_facility_type,number_of_beds,total_bed_days_available,occupancy_rate,total_discharges__v___xviii___xix___unknown_,total_days__v___xviii___xix___unknown_,fte___employees_on_payroll,...,population,median_age,median_income,poverty_rate,higher_education_rate,Other,wage_physician,wage_practitioner,wage_rn,wage_support
0,2011,AL,General Short-Term (includes CAH),Short-Term Hospital,114.0,41610.0,0.472026,5283.0,19641.0,598.72,...,NaN,NaN,NaN,NaN,NaN,53754.888134,200193.678647,78568.610526,57840.0,25776.710446
1,2011,MT,General Short-Term (includes CAH),Critical Access Hospital,25.0,9125.0,0.131068,155.0,1196.0,70.56,...,NaN,NaN,NaN,NaN,NaN,53324.679487,181134.576271,68750.470914,60190.0,27864.492140
2,2011,AL,General Short-Term (includes CAH),Short-Term Hospital,46.0,16790.0,0.159678,892.0,2681.0,72.65,...,NaN,NaN,NaN,NaN,NaN,53754.888134,200193.678647,78568.610526,57840.0,25776.710446
3,2011,AL,Rehabilitation Hospital,Rehabilitation Hospital,100.0,36500.0,0.887068,2390.0,32378.0,297.77,...,NaN,NaN,NaN,NaN,NaN,53754.888134,200193.678647,78568.610526,57840.0,25776.710446
4,2011,FL,Rehabilitation Hospital,Rehabilitation Hospital,70.0,25550.0,0.663836,1370.0,16961.0,154.60,...,NaN,NaN,NaN,NaN,NaN,61780.959858,186447.328473,78132.768107,64020.0,28961.926688


In [9]:
hospital.shape

(75870, 56)

In [11]:
#  Save the final dataset
hospital.to_csv(
    r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\ML Modelling\hospital_cms_census_bls_rucc.csv",
    index=False,
)

In [14]:
# Census-related columns
census_cols = [
    "population",
    "median_age",
    "median_income",
    "poverty_rate",
    "higher_education_rate"
]

# Missing rate by year
missing_by_year = (
    hospital
    .groupby("year")[census_cols]
    .apply(lambda x: x.isna().mean())
)

print(missing_by_year)


      population  median_age  median_income  poverty_rate  \
year                                                        
2011    1.000000    1.000000       1.000000      1.000000   
2012    0.064673    0.064673       0.064673      0.064673   
2013    0.062061    0.062061       0.062061      0.062061   
2014    0.062724    0.062724       0.062724      0.062724   
2015    0.062597    0.062597       0.062597      0.062597   
2016    0.078650    0.078650       0.078650      0.078650   
2017    0.096412    0.096412       0.096412      0.096412   
2018    0.102864    0.102864       0.103023      0.103023   
2019    0.106733    0.106733       0.106733      0.106733   
2020    0.111772    0.111772       0.111772      0.111772   
2021    0.113062    0.113062       0.113062      0.113062   
2022    0.114580    0.114580       0.114580      0.114580   

      higher_education_rate  
year                         
2011               1.000000  
2012               0.064673  
2013               0.0620